# 03 — Model Comparison

Train all six models and compare. For the full run use `python train.py`; this notebook reloads the saved comparison or runs a quick subset.

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import config
from src import ingestion, preprocessing, features, models
from src.evaluate import compare_models, plot_predictions

df = ingestion.load_or_generate()
df = preprocessing.handle_missing_values(df); df = preprocessing.drop_long_gaps(df)
df = preprocessing.remove_outliers(df, config.POLLUTANT_COLS + [config.TARGET_COL])
feat = features.build_features(df); cols = features.get_feature_columns(feat)
train_df, test_df = preprocessing.train_test_split_temporal(feat)
train_s, scaler = preprocessing.normalize_features(train_df.copy(), cols)
test_s = test_df.copy(); test_s[cols] = scaler.transform(test_df[cols])

In [ ]:
results, fitted = models.train_all_models(train_s[cols], train_s[config.TARGET_COL],
                                          test_s[cols], test_s[config.TARGET_COL], test_s)
compare_models(results)

In [ ]:
best = min((k for k in results if fitted.get(k) is not None), key=lambda k: results[k]['RMSE'])
preds = fitted[best].predict(test_s[cols])
plot_predictions(test_s[config.TARGET_COL], preds, best)